# Llama 2 7B Nepali Multi-Dataset QLoRA

Fine-tune the smallest official Llama 2 checkpoint, `meta-llama/Llama-2-7b-chat-hf`, with a selectable source:

- `alpaca_nepali`: `saillab/alpaca-nepali-cleaned` from Hugging Face. Its text is not cleaned again; it is only converted to conversational prompt/completion records.
- `lima_translated`: the repository's `data_generation_pipeline/lima_translations.json`. Tagged `HUMAN`/`ASSISTANT` text is parsed and passed through the same Nepali cleaning primitives used by the codebase pipeline.

The common normalization boundary also supports custom field maps, prompt/completion data, and common chat schemas. The notebook audits rejected rows before training, runs a base-model baseline, performs 4-bit QLoRA, plots metrics, saves the adapter and run manifest, and compares the fine-tuned output with the saved baseline.

## 1. Prerequisites

Accept Meta's Llama 2 terms on Hugging Face and expose an authorized `HF_TOKEN`. This notebook requires an NVIDIA CUDA GPU; a T4 can run the default 512-token, batch-size-one QLoRA setup, while a larger GPU allows longer contexts or batches. Run from this repository (or set `ATTENTION_MAPS_ROOT`) because translated-LIMA preprocessing imports the repository pipeline utilities.

In [ ]:
%pip install -q -U transformers datasets accelerate bitsandbytes peft trl huggingface_hub "matplotlib<3.11"

## 2. Imports, repository discovery, and reproducibility

In [ ]:
import inspect
import json
import math
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import transformers
import trl
from datasets import load_dataset
from huggingface_hub import login
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

def find_repository_root():
    configured = os.getenv('ATTENTION_MAPS_ROOT')
    candidates = ([Path(configured)] if configured else []) + [
        Path.cwd(),
        *Path.cwd().parents,
        Path('/kaggle/working/attentionMaps'),
    ]
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'attention_maps').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not find the attentionMaps repository. Set ATTENTION_MAPS_ROOT.'
    )

PROJECT_ROOT = find_repository_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from attention_maps.training.sft_data import normalize_sft_example, register_sft_schema
from scripts.utils.nepali_text import CleaningConfig

SEED = 42
set_seed(SEED)
print('Repository:', PROJECT_ROOT)
print('Transformers:', transformers.__version__)
print('TRL:', trl.__version__)

## 3. Choose the dataset and training configuration

Change only `DATASET_SOURCE` for the two built-in paths. To add a future schema, add a source configuration and set `FIELD_MAP` from canonical names (`instruction`, `input`, `output`, `translation`, `prompt`, `completion`, `messages`, or `conversations`) to its physical columns. Use `schema='auto'` when the source already resembles a supported layout.

In [ ]:
MODEL_NAME = 'meta-llama/Llama-2-7b-chat-hf'
DATASET_SOURCE = 'alpaca_nepali'  # 'alpaca_nepali' or 'lima_translated'

DATASET_SOURCES = {
    'alpaca_nepali': {
        'kind': 'huggingface',
        'dataset_id': 'saillab/alpaca-nepali-cleaned',
        'split': 'train',
        'schema': 'alpaca',
        'field_map': None,
        'apply_pipeline_cleaning': False,
    },
    'lima_translated': {
        'kind': 'json',
        'path': PROJECT_ROOT / 'data_generation_pipeline' / 'lima_translations.json',
        'schema': 'lima',
        'field_map': None,
        'apply_pipeline_cleaning': True,
    },
}

# Optional override for a future dataset, for example:
# FIELD_MAP = {'instruction': 'question', 'input': 'context', 'output': 'answer'}
FIELD_MAP = None
MAX_LENGTH = 512
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
TEST_SIZE = 0.02
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else PROJECT_ROOT / 'runs'
OUTPUT_DIR = WORK_DIR / f'llama2-7b-{DATASET_SOURCE}-qlora'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATASET_SOURCE not in DATASET_SOURCES:
    raise ValueError(f'Unknown DATASET_SOURCE: {DATASET_SOURCE}')
SOURCE_CONFIG = DATASET_SOURCES[DATASET_SOURCE]
print(json.dumps({
    key: str(value) if isinstance(value, Path) else value
    for key, value in SOURCE_CONFIG.items()
}, indent=2))

## 4. Hugging Face access and GPU precision

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN') or os.getenv('HF_token')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print('WARNING: HF_TOKEN is missing; the gated Llama 2 model will not load.')

if not torch.cuda.is_available():
    raise RuntimeError('This QLoRA notebook requires an NVIDIA CUDA GPU.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MAJOR, GPU_MINOR = torch.cuda.get_device_capability(0)
USE_BF16 = GPU_MAJOR >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print('GPU:', GPU_NAME)
print('Compute capability:', GPU_MAJOR, GPU_MINOR)
print('QLoRA compute dtype:', COMPUTE_DTYPE)

## 5. Load the selected source

The Hub Alpaca source is loaded as supplied. The local JSON source is read through `datasets`, so normalization remains identical after a future move to JSONL or multiple shards.

In [ ]:
if SOURCE_CONFIG['kind'] == 'huggingface':
    raw_dataset = load_dataset(
        SOURCE_CONFIG['dataset_id'],
        split=SOURCE_CONFIG.get('split', 'train'),
        token=HF_TOKEN,
    )
elif SOURCE_CONFIG['kind'] in {'json', 'parquet'}:
    source_path = Path(SOURCE_CONFIG['path'])
    if not source_path.is_file():
        raise FileNotFoundError(f'Dataset file not found: {source_path}')
    raw_dataset = load_dataset(
        SOURCE_CONFIG['kind'],
        data_files=str(source_path),
        split='train',
    )
else:
    raise ValueError(f"Unsupported source kind: {SOURCE_CONFIG['kind']}")

print(raw_dataset)
print('Columns:', raw_dataset.column_names)
print('First raw record:', raw_dataset[0])

## 6. Normalize, audit, and split

`cleaning_config=None` means the Alpaca text is not reprocessed. Translated LIMA uses preserve-mode NFC normalization, markup/control/URL removal, and a deliberately relaxed Devanagari threshold so technical answers and code examples are retained. Rejections are counted before they are removed.

In [ ]:
cleaning_config = None
if SOURCE_CONFIG.get('apply_pipeline_cleaning'):
    cleaning_config = CleaningConfig(
        min_devanagari_ratio=0.05,
        min_devanagari_letters=1,
        min_characters=1,
        mode='preserve',
        normalization='NFC',
    )

def normalize_record(example):
    return normalize_sft_example(
        example,
        schema=SOURCE_CONFIG.get('schema', 'auto'),
        field_map=FIELD_MAP or SOURCE_CONFIG.get('field_map'),
        cleaning_config=cleaning_config,
    )

normalized_dataset = raw_dataset.map(
    normalize_record,
    remove_columns=raw_dataset.column_names,
    desc='Normalizing SFT records',
)
status_counts = Counter(normalized_dataset['normalization_status'])
print('Normalization audit:', json.dumps(status_counts, indent=2, ensure_ascii=False))

conversation_dataset = normalized_dataset.filter(
    lambda row: row['normalization_status'] == 'accepted',
    desc='Removing rejected records',
).remove_columns(['source_schema', 'normalization_status'])
if len(conversation_dataset) < 2:
    raise RuntimeError('Need at least two accepted examples for train/evaluation splits.')
dataset_split = conversation_dataset.train_test_split(
    test_size=TEST_SIZE,
    seed=SEED,
)
print(dataset_split)
print(json.dumps(dataset_split['train'][0], ensure_ascii=False, indent=2))

## 7. Load the tokenizer and 4-bit base model

NF4 plus double quantization keeps the frozen 7B base compact. Trainable LoRA parameters are later kept in FP32 for stable AMP behavior.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map={'': 0},
    dtype=COMPUTE_DTYPE,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = True
model.config.pretraining_tp = 1
print('pad_token:', repr(tokenizer.pad_token))
print('pad_token_id:', tokenizer.pad_token_id)

## 8. Save a base-model inference baseline

In [ ]:
SAMPLE_MESSAGES = dataset_split['test'][0]['prompt']

@torch.inference_mode()
def generate_response(model, tokenizer, messages, max_new_tokens=128):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(
        rendered,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)
    torch.manual_seed(SEED)
    generated = model.generate(
        **encoded,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    answer_ids = generated[0, encoded['input_ids'].shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

model.eval()
base_answer = generate_response(model, tokenizer, SAMPLE_MESSAGES)
print('PROMPT:', json.dumps(SAMPLE_MESSAGES, ensure_ascii=False, indent=2))
print('BASE MODEL RESPONSE:\n', base_answer)

## 9. Attach QLoRA adapters

In [ ]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model = get_peft_model(model, lora_config, autocast_adapter_dtype=True)
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()
trainable_dtypes = Counter(
    str(parameter.dtype) for parameter in model.parameters()
    if parameter.requires_grad
)
assert set(trainable_dtypes) == {'torch.float32'}, trainable_dtypes
model.print_trainable_parameters()
print('Trainable dtypes:', trainable_dtypes)

## 10. Metric history callback and version-tolerant SFT configuration

In [ ]:
class TrainingCurveCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = Path(output_dir)
        self.history = []
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_world_process_zero and logs:
            record = {'step': int(state.global_step), 'epoch': state.epoch}
            record.update({
                key: float(value) for key, value in logs.items()
                if isinstance(value, (int, float))
            })
            self.history.append(record)
            (self.output_dir / 'training_metrics.json').write_text(
                json.dumps(self.history, indent=2), encoding='utf-8'
            )

    def plot(self):
        train = [row for row in self.history if 'loss' in row]
        evaluation = [row for row in self.history if 'eval_loss' in row]
        if not train and not evaluation:
            return None
        figure, axis = plt.subplots(figsize=(9, 5), dpi=140)
        if train:
            axis.plot([row['step'] for row in train], [row['loss'] for row in train], label='train')
        if evaluation:
            axis.plot([row['step'] for row in evaluation], [row['eval_loss'] for row in evaluation], label='eval')
        axis.set(xlabel='optimizer step', ylabel='cross-entropy loss', title='Llama 2 Nepali QLoRA')
        axis.grid(alpha=0.25)
        axis.legend()
        figure.tight_layout()
        path = self.output_dir / 'training_curves.png'
        figure.savefig(path, bbox_inches='tight')
        plt.show()
        return path

curve_callback = TrainingCurveCallback(OUTPUT_DIR)
device_count = max(1, torch.cuda.device_count())
effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * device_count
steps_per_epoch = math.ceil(len(dataset_split['train']) / effective_batch_size)
total_steps = max(1, math.ceil(steps_per_epoch * NUM_EPOCHS))
eval_steps = max(1, min(200, total_steps // 10))

config_values = dict(
    output_dir=str(OUTPUT_DIR),
    max_length=MAX_LENGTH,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=max(1, int(0.03 * total_steps)),
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=1.0,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=eval_steps,
    eval_accumulation_steps=4,
    save_strategy='steps',
    save_steps=eval_steps,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    packing=False,
    completion_only_loss=True,
)
supported = inspect.signature(SFTConfig.__init__).parameters
filtered_values = {key: value for key, value in config_values.items() if key in supported}
sft_config = SFTConfig(**filtered_values)
print('Effective batch size:', effective_batch_size)
print('Expected optimizer steps:', total_steps)
print('Evaluate/save every:', eval_steps, 'steps')
print('Ignored unsupported settings:', sorted(set(config_values) - set(filtered_values)))

## 11. Create the trainer and run preflight checks

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset_split['train'],
    eval_dataset=dataset_split['test'],
    processing_class=tokenizer,
    callbacks=[curve_callback],
)
trainer.can_return_loss = True
trainer.compute_metrics = None
trainer.preprocess_logits_for_metrics = None
for parameter in trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.float()
        parameter.grad = None
trainer_dtypes = {
    parameter.dtype for parameter in trainer.model.parameters()
    if parameter.requires_grad
}
assert trainer_dtypes == {torch.float32}, trainer_dtypes
assert len(trainer.train_dataset) > 0
print('Prepared training rows:', len(trainer.train_dataset))
print('Prepared evaluation rows:', len(trainer.eval_dataset))
initial_metrics = trainer.evaluate()
print('Pre-training evaluation:', initial_metrics)
if not math.isfinite(float(initial_metrics.get('eval_loss', float('nan')))):
    raise RuntimeError('Non-finite initial evaluation loss; stop before training.')

## 12. Train, evaluate, save adapter, and record provenance

In [ ]:
last_checkpoint = None  # Set a known-good checkpoint path to resume explicitly.
train_result = trainer.train(resume_from_checkpoint=last_checkpoint or False)
final_metrics = trainer.evaluate()
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)
trainer.log_metrics('final_eval', final_metrics)
trainer.save_metrics('final_eval', final_metrics)
trainer.save_state()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

run_manifest = {
    'base_model': MODEL_NAME,
    'dataset_source': DATASET_SOURCE,
    'source_config': {
        key: str(value) if isinstance(value, Path) else value
        for key, value in SOURCE_CONFIG.items()
    },
    'field_map': FIELD_MAP,
    'pipeline_cleaning_applied': cleaning_config is not None,
    'normalization_audit': dict(status_counts),
    'train_rows': len(dataset_split['train']),
    'evaluation_rows': len(dataset_split['test']),
    'seed': SEED,
    'max_length': MAX_LENGTH,
}
(OUTPUT_DIR / 'run_manifest.json').write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Final evaluation:', final_metrics)
print('Saved adapter and manifest to:', OUTPUT_DIR)
curve_callback.plot()

## 13. Compare base and fine-tuned responses

In [ ]:
trainer.model.eval()
trainer.model.config.use_cache = True
fine_tuned_answer = generate_response(trainer.model, tokenizer, SAMPLE_MESSAGES)
print('=' * 80)
print('PROMPT:\n', json.dumps(SAMPLE_MESSAGES, ensure_ascii=False, indent=2))
print('=' * 80)
print('BASE MODEL RESPONSE:\n', base_answer)
print('=' * 80)
print('FINE-TUNED MODEL RESPONSE:\n', fine_tuned_answer)

## 14. Reload the adapter in a fresh session

In [ ]:
# Run after the imports/configuration/model-quantization cells in a fresh runtime.
reload_tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR))
reload_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map={'': 0},
    dtype=COMPUTE_DTYPE,
)
reloaded_model = PeftModel.from_pretrained(reload_base, str(OUTPUT_DIR))
reloaded_model.eval()
print(generate_response(reloaded_model, reload_tokenizer, SAMPLE_MESSAGES))

## Adding tomorrow's dataset

1. Add a source entry in `DATASET_SOURCES` and load its format in Section 5.
2. Prefer `schema='auto'` for Alpaca, translated LIMA, prompt/completion, `messages`, or `conversations` records.
3. For renamed columns, set `FIELD_MAP`, e.g. `{'instruction': 'question', 'input': 'context', 'output': 'answer'}`.
4. For a structurally new schema, register a small callable with `register_sft_schema` in `attention_maps.training.sft_data`; the downstream audit, split, trainer, and provenance logic remain unchanged.
5. Enable pipeline cleaning only when the source has not already been cleaned. Always inspect the rejection audit and sample before training.